In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
from datetime import datetime
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.linear_model import Ridge
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("RAINFALL PREDICTION MODEL TRAINING AND SAVING")
print("="*80)

# Load data
df = pd.read_csv('TIA-WEATHER-DATA-2015-01-01-TO-2025-06-30 (1).csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime')

print(f"Dataset loaded: {len(df)} records")
print(f"Date range: {df['datetime'].min()} to {df['datetime'].max()}")

# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

# Basic features
df['temp_range'] = df['tempmax'] - df['tempmin']
df['dew_point_depression'] = df['temp'] - df['dew']
df['humidity_deficit'] = 100 - df['humidity']
df['temp_dew_ratio'] = df['temp'] / (df['dew'] + 0.01)

# Atmospheric features
df['pressure_trend'] = df['sealevelpressure'].diff(1)
df['pressure_trend_3'] = df['sealevelpressure'].diff(3)
df['pressure_acceleration'] = df['pressure_trend'].diff(1)

# Wind features
df['wind_power'] = df['windspeed'] ** 3
df['wind_gust_ratio'] = df['windgust'] / (df['windspeed'] + 0.01)
df['wind_energy'] = 0.5 * df['windspeed'] ** 2

# Cloud features
df['cloud_cover_ratio'] = df['cloudcover'] / 100
df['solar_efficiency'] = df['solarenergy'] / (df['solarradiation'] + 0.01)
df['uv_radiation_ratio'] = df['uvindex'] / (df['solarradiation'] + 0.01)

# Time features
df['month'] = df['datetime'].dt.month
df['dayofyear'] = df['datetime'].dt.dayofyear

# Cyclical features
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['day_sin'] = np.sin(2 * np.pi * df['dayofyear'] / 365)
df['day_cos'] = np.cos(2 * np.pi * df['dayofyear'] / 365)

# Interaction features
df['temp_humidity_interaction'] = df['temp'] * df['humidity']
df['pressure_cloud_interaction'] = df['sealevelpressure'] * df['cloudcover']
df['wind_cloud_interaction'] = df['windspeed'] * df['cloudcover']

# Lag features
for lag in [1, 2, 3, 7, 14]:
    df[f'precip_lag_{lag}'] = df['precipitation'].shift(lag)
    df[f'temp_lag_{lag}'] = df['temp'].shift(lag)
    df[f'humidity_lag_{lag}'] = df['humidity'].shift(lag)
    df[f'pressure_lag_{lag}'] = df['sealevelpressure'].shift(lag)
    df[f'cloud_lag_{lag}'] = df['cloudcover'].shift(lag)
    df[f'wind_lag_{lag}'] = df['windspeed'].shift(lag)

# Rolling statistics
for window in [3, 7, 14]:
    df[f'temp_roll_mean_{window}'] = df['temp'].rolling(window=window, min_periods=1).mean()
    df[f'temp_roll_std_{window}'] = df['temp'].rolling(window=window, min_periods=1).std()
    df[f'humidity_roll_mean_{window}'] = df['humidity'].rolling(window=window, min_periods=1).mean()
    df[f'pressure_roll_mean_{window}'] = df['sealevelpressure'].rolling(window=window, min_periods=1).mean()
    df[f'precip_roll_sum_{window}'] = df['precipitation'].rolling(window=window, min_periods=1).sum()
    df[f'cloud_roll_mean_{window}'] = df['cloudcover'].rolling(window=window, min_periods=1).mean()

# Rate of change
df['temp_rate_change'] = df['temp'].pct_change(1)
df['pressure_rate_change'] = df['sealevelpressure'].pct_change(1)
df['humidity_rate_change'] = df['humidity'].pct_change(1)

# Anomaly features
df['temp_anomaly_7'] = df['temp'] - df['temp_roll_mean_7']
df['pressure_anomaly_7'] = df['sealevelpressure'] - df['pressure_roll_mean_7']

# Target
df['precipitation_log'] = np.log1p(df['precipitation'])

print(f"Features created: {len(df.columns)}")

# ============================================================================
# SELECT FEATURES FOR MODELING
# ============================================================================

print("\n" + "="*60)
print("FEATURE SELECTION")
print("="*60)

# Define feature list (manually curated based on importance)
selected_features = [
    # Primary weather variables
    'temp', 'tempmax', 'tempmin', 'dew', 'humidity', 'precipprob', 
    'precipcover', 'windgust', 'windspeed', 'winddir', 
    'sealevelpressure', 'cloudcover', 'visibility', 
    'solarradiation', 'solarenergy', 'uvindex',
    
    # Engineered features
    'temp_range', 'dew_point_depression', 'humidity_deficit', 'temp_dew_ratio',
    'pressure_trend', 'pressure_trend_3', 'wind_power', 'wind_gust_ratio',
    'cloud_cover_ratio', 'temp_humidity_interaction', 'pressure_cloud_interaction',
    
    # Time features
    'month_sin', 'month_cos', 'day_sin', 'day_cos',
    
    # Lag features (recent)
    'precip_lag_1', 'precip_lag_2', 'precip_lag_3',
    'temp_lag_1', 'temp_lag_2', 'temp_lag_3',
    'humidity_lag_1', 'humidity_lag_2', 'humidity_lag_3',
    'pressure_lag_1', 'pressure_lag_2',
    
    # Rolling statistics
    'temp_roll_mean_3', 'temp_roll_std_3',
    'humidity_roll_mean_3', 'pressure_roll_mean_3',
    'precip_roll_sum_3', 'cloud_roll_mean_3',
    
    # Rate of change
    'temp_rate_change', 'pressure_rate_change',
    'temp_anomaly_7', 'pressure_anomaly_7'
]

# Filter only existing features
selected_features = [f for f in selected_features if f in df.columns]
print(f"Selected {len(selected_features)} features for modeling")

# ============================================================================
# PREPARE DATA FOR REGRESSION (RAINY DAYS ONLY)
# ============================================================================

print("\n" + "="*60)
print("DATA PREPARATION")
print("="*60)

# Use only rainy days for regression
reg_mask = df['precipitation'] > 0
df_reg = df[reg_mask].copy()

print(f"Total records: {len(df):,}")
print(f"Rainy days: {len(df_reg):,} ({len(df_reg)/len(df)*100:.1f}%)")

# Prepare features and target
X = df_reg[selected_features].fillna(df_reg[selected_features].median())
y = df_reg['precipitation_log']

# Time-based split (80/20 chronological)
split_idx = int(len(df_reg) * 0.8)
train_idx = df_reg.index[:split_idx]
test_idx = df_reg.index[split_idx:]

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]
y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

print(f"Training samples: {len(X_train):,}")
print(f"Test samples: {len(X_test):,}")

# ============================================================================
# SCALE FEATURES
# ============================================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeatures scaled successfully")

# ============================================================================
# TRAIN RANDOM FOREST MODEL
# ============================================================================

print("\n" + "="*60)
print("TRAINING RANDOM FOREST MODEL")
print("="*60)

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_model.fit(X_train_scaled, y_train)

# Evaluate Random Forest
y_pred_rf_log = rf_model.predict(X_test_scaled)
y_pred_rf = np.expm1(y_pred_rf_log)
y_actual = np.expm1(y_test)

rmse_rf = np.sqrt(mean_squared_error(y_actual, y_pred_rf))
mae_rf = mean_absolute_error(y_actual, y_pred_rf)
r2_rf = r2_score(y_actual, y_pred_rf)

print(f"\nRandom Forest Performance:")
print(f"  RMSE: {rmse_rf:.2f} mm")
print(f"  MAE: {mae_rf:.2f} mm")
print(f"  R²: {r2_rf:.4f}")

# ============================================================================
# TRAIN LIGHTGBM MODEL
# ============================================================================

print("\n" + "="*60)
print("TRAINING LIGHTGBM MODEL")
print("="*60)

lgbm_model = LGBMRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,
    reg_lambda=0.01,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgbm_model.fit(X_train_scaled, y_train)

# Evaluate LightGBM
y_pred_lgbm_log = lgbm_model.predict(X_test_scaled)
y_pred_lgbm = np.expm1(y_pred_lgbm_log)

rmse_lgbm = np.sqrt(mean_squared_error(y_actual, y_pred_lgbm))
mae_lgbm = mean_absolute_error(y_actual, y_pred_lgbm)
r2_lgbm = r2_score(y_actual, y_pred_lgbm)

print(f"\nLightGBM Performance:")
print(f"  RMSE: {rmse_lgbm:.2f} mm")
print(f"  MAE: {mae_lgbm:.2f} mm")
print(f"  R²: {r2_lgbm:.4f}")

# ============================================================================
# TRAIN XGBOOST MODEL (for stacking ensemble)
# ============================================================================

print("\n" + "="*60)
print("TRAINING XGBOOST MODEL")
print("="*60)

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.01,
    reg_lambda=0.01,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train)

# Evaluate XGBoost
y_pred_xgb_log = xgb_model.predict(X_test_scaled)
y_pred_xgb = np.expm1(y_pred_xgb_log)

rmse_xgb = np.sqrt(mean_squared_error(y_actual, y_pred_xgb))
mae_xgb = mean_absolute_error(y_actual, y_pred_xgb)
r2_xgb = r2_score(y_actual, y_pred_xgb)

print(f"\nXGBoost Performance:")
print(f"  RMSE: {rmse_xgb:.2f} mm")
print(f"  MAE: {mae_xgb:.2f} mm")
print(f"  R²: {r2_xgb:.4f}")

# ============================================================================
# TRAIN GRADIENT BOOSTING MODEL (for stacking ensemble)
# ============================================================================

print("\n" + "="*60)
print("TRAINING GRADIENT BOOSTING MODEL")
print("="*60)

gb_model = GradientBoostingRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    random_state=42
)

gb_model.fit(X_train_scaled, y_train)

# Evaluate Gradient Boosting
y_pred_gb_log = gb_model.predict(X_test_scaled)
y_pred_gb = np.expm1(y_pred_gb_log)

rmse_gb = np.sqrt(mean_squared_error(y_actual, y_pred_gb))
mae_gb = mean_absolute_error(y_actual, y_pred_gb)
r2_gb = r2_score(y_actual, y_pred_gb)

print(f"\nGradient Boosting Performance:")
print(f"  RMSE: {rmse_gb:.2f} mm")
print(f"  MAE: {mae_gb:.2f} mm")
print(f"  R²: {r2_gb:.4f}")

# ============================================================================
# TRAIN STACKING ENSEMBLE
# ============================================================================

print("\n" + "="*60)
print("TRAINING STACKING ENSEMBLE")
print("="*60)

# Define base models for stacking
base_models = [
    ('rf', RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )),
    ('lgbm', LGBMRegressor(
        n_estimators=300,
        max_depth=7,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )),
    ('xgb', XGBRegressor(
        n_estimators=300,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    ))
]

# Create stacking ensemble with Ridge as meta-learner
stacking_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    passthrough=False,
    n_jobs=-1
)

print("Training stacking ensemble (this may take a few minutes)...")
stacking_model.fit(X_train_scaled, y_train)

# Evaluate Stacking Ensemble
y_pred_stack_log = stacking_model.predict(X_test_scaled)
y_pred_stack = np.expm1(y_pred_stack_log)

rmse_stack = np.sqrt(mean_squared_error(y_actual, y_pred_stack))
mae_stack = mean_absolute_error(y_actual, y_pred_stack)
r2_stack = r2_score(y_actual, y_pred_stack)

print(f"\nStacking Ensemble Performance:")
print(f"  RMSE: {rmse_stack:.2f} mm")
print(f"  MAE: {mae_stack:.2f} mm")
print(f"  R²: {r2_stack:.4f}")

# ============================================================================
# SAVE MODELS AND PREPROCESSING OBJECTS
# ============================================================================

print("\n" + "="*60)
print("SAVING MODELS AND PREPROCESSING OBJECTS")
print("="*60)

# Save models
joblib.dump(rf_model, 'random_forest_model.pkl')
joblib.dump(lgbm_model, 'lightgbm_model.pkl')
joblib.dump(stacking_model, 'stacking_ensemble_model.pkl')
joblib.dump(xgb_model, 'xgboost_model.pkl')
joblib.dump(gb_model, 'gradient_boosting_model.pkl')

# Save preprocessing objects
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(selected_features, 'selected_features.pkl')

print("\n✓ Models saved successfully:")
print("  - random_forest_model.pkl")
print("  - lightgbm_model.pkl")
print("  - stacking_ensemble_model.pkl")
print("  - xgboost_model.pkl")
print("  - gradient_boosting_model.pkl")
print("\n✓ Preprocessing objects saved:")
print("  - scaler.pkl")
print("  - selected_features.pkl")

# ============================================================================
# CREATE A SUMMARY REPORT
# ============================================================================

print("\n" + "="*80)
print("MODEL PERFORMANCE SUMMARY")
print("="*80)

# Create comparison DataFrame
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'LightGBM', 'XGBoost', 'Gradient Boosting', 'Stacking Ensemble'],
    'RMSE (mm)': [rmse_rf, rmse_lgbm, rmse_xgb, rmse_gb, rmse_stack],
    'MAE (mm)': [mae_rf, mae_lgbm, mae_xgb, mae_gb, mae_stack],
    'R² Score': [r2_rf, r2_lgbm, r2_xgb, r2_gb, r2_stack]
})
comparison = comparison.sort_values('R² Score', ascending=False)

print("\n" + comparison.to_string(index=False))
print("\n" + "="*80)

# ============================================================================
# VERIFICATION CODE (Test loading saved models)
# ============================================================================

print("\n" + "="*60)
print("VERIFYING SAVED MODELS")
print("="*60)

# Test loading models
loaded_rf = joblib.load('random_forest_model.pkl')
loaded_lgbm = joblib.load('lightgbm_model.pkl')
loaded_stack = joblib.load('stacking_ensemble_model.pkl')
loaded_scaler = joblib.load('scaler.pkl')
loaded_features = joblib.load('selected_features.pkl')

print("\n✓ All models loaded successfully!")
print(f"✓ Features expected: {len(loaded_features)}")
print(f"✓ Random Forest is a {type(loaded_rf).__name__}")
print(f"✓ LightGBM is a {type(loaded_lgbm).__name__}")
print(f"✓ Stacking Ensemble is a {type(loaded_stack).__name__}")

# Test prediction with sample data
sample_data = X_test.iloc[[0]]
sample_scaled = loaded_scaler.transform(sample_data)

pred_rf = loaded_rf.predict(sample_scaled)
pred_lgbm = loaded_lgbm.predict(sample_scaled)
pred_stack = loaded_stack.predict(sample_scaled)

print(f"\n✓ Sample predictions successful!")
print(f"  Random Forest prediction: {np.expm1(pred_rf[0]):.2f} mm")
print(f"  LightGBM prediction: {np.expm1(pred_lgbm[0]):.2f} mm")
print(f"  Stacking Ensemble prediction: {np.expm1(pred_stack[0]):.2f} mm")

print("\n" + "="*80)
print("MODEL TRAINING AND SAVING COMPLETED SUCCESSFULLY!")
print("="*80)

RAINFALL PREDICTION MODEL TRAINING AND SAVING
Dataset loaded: 3468 records
Date range: 2015-01-01 00:00:00 to 2025-06-30 00:00:00

FEATURE ENGINEERING
Features created: 98

FEATURE SELECTION
Selected 52 features for modeling

DATA PREPARATION
Total records: 3,468
Rainy days: 1,361 (39.2%)
Training samples: 1,088
Test samples: 273

Features scaled successfully

TRAINING RANDOM FOREST MODEL


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 168 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:    0.3s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 300 out of 300 | elapsed:    0.0s finished



Random Forest Performance:
  RMSE: 15.09 mm
  MAE: 7.14 mm
  R²: 0.2649

TRAINING LIGHTGBM MODEL

LightGBM Performance:
  RMSE: 9.47 mm
  MAE: 3.22 mm
  R²: 0.7109

TRAINING XGBOOST MODEL

XGBoost Performance:
  RMSE: 11.42 mm
  MAE: 4.59 mm
  R²: 0.5795

TRAINING GRADIENT BOOSTING MODEL

Gradient Boosting Performance:
  RMSE: 10.10 mm
  MAE: 3.14 mm
  R²: 0.6706

TRAINING STACKING ENSEMBLE
Training stacking ensemble (this may take a few minutes)...

Stacking Ensemble Performance:
  RMSE: 9.53 mm
  MAE: 3.38 mm
  R²: 0.7068

SAVING MODELS AND PREPROCESSING OBJECTS

✓ Models saved successfully:
  - random_forest_model.pkl
  - lightgbm_model.pkl
  - stacking_ensemble_model.pkl
  - xgboost_model.pkl
  - gradient_boosting_model.pkl

✓ Preprocessing objects saved:
  - scaler.pkl
  - selected_features.pkl

MODEL PERFORMANCE SUMMARY

            Model  RMSE (mm)  MAE (mm)  R² Score
         LightGBM   9.466914  3.216052  0.710862
Stacking Ensemble   9.532716  3.377061  0.706829
Gradient Boos

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 168 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 300 out of 300 | elapsed:    0.0s finished
